# Exercises

In this section we have two exercises:
1. Implement the polynomial kernel.
2. Implement the multiclass C-SVM.

## Polynomial kernel

You need to extend the ``build_kernel`` function and implement the polynomial kernel if the ``kernel_type`` is set to 'poly'. The equation that needs to be implemented:
\begin{equation}
K=(X^{T}*Y)^{d}.
\end{equation}

In [5]:
def build_kernel(data_set, kernel_type='linear'):
    kernel = np.dot(data_set, data_set.T)
    if kernel_type == 'rbf':
        sigma = 1.0
        objects_count = len(data_set)
        b = np.ones((len(data_set), 1))
        kernel -= 0.5 * (np.dot((np.diag(kernel)*np.ones((1, objects_count))).T, b.T)
                         + np.dot(b, (np.diag(kernel) * np.ones((1, objects_count))).T.T))
        kernel = np.exp(kernel / (2. * sigma ** 2))
    elif kernel_type == 'poly':
        kernel = (kernel + 1) ** degree
    return kernel

## Implement a multiclass C-SVM

Use the classification method that we used in notebook 7.3 and IRIS dataset to build a multiclass C-SVM classifier. Most implementation is about a function that will return the proper data set that need to be used for the prediction. You need to implement:
- ``choose_set_for_label``
- ``get_labels_count``

In [6]:
def choose_set_for_label(data_set, labels, target_label):
    binary_labels = np.array([1 if l == target_label else -1 for l in labels])
    return train_test_split(data_set, binary_labels, test_size=0.2, random_state=15)

In [7]:
def get_labels_count(data_set):
    return labels_count

In [8]:
def train(train_data_set, train_labels, kernel_type='linear', C=10, threshold=1e-5):
    kernel = build_kernel(train_data_set, kernel_type=kernel_type)

    P = train_labels * train_labels.transpose() * kernel
    q = -np.ones((objects_count, 1))
    G = np.concatenate((np.eye(objects_count), -np.eye(objects_count)))
    h = np.concatenate((C * np.ones((objects_count, 1)), np.zeros((objects_count, 1))))

    A = train_labels.reshape(1, objects_count)
    A = A.astype(float)
    b = 0.0

    sol = cvxopt.solvers.qp(cvxopt.matrix(P), cvxopt.matrix(q), cvxopt.matrix(G), cvxopt.matrix(h), cvxopt.matrix(A), cvxopt.matrix(b))

    lambdas = np.array(sol['x'])

    support_vectors_id = np.where(lambdas > threshold)[0]
    vector_number = len(support_vectors_id)
    support_vectors = train_data_set[support_vectors_id, :]

    lambdas = lambdas[support_vectors_id]
    targets = train_labels[support_vectors_id]

    b = np.sum(targets)
    for n in range(vector_number):
        b -= np.sum(lambdas * targets * np.reshape(kernel[support_vectors_id[n], support_vectors_id], (vector_number, 1)))
    b /= len(lambdas)

    return lambdas, support_vectors, support_vectors_id, b, targets, vector_number

def build_kernel(data_set, kernel_type='linear'):
    kernel = np.dot(data_set, data_set.T)
    if kernel_type == 'rbf':
        sigma = 1.0
        objects_count = len(data_set)
        b = np.ones((len(data_set), 1))
        kernel -= 0.5 * (np.dot((np.diag(kernel)*np.ones((1, objects_count))).T, b.T)
                         + np.dot(b, (np.diag(kernel) * np.ones((1, objects_count))).T.T))
        kernel = np.exp(kernel / (2. * sigma ** 2))
    return kernel

def classify_rbf(test_data_set, train_data_set, lambdas, targets, b, vector_number, support_vectors, support_vectors_id):
    kernel = np.dot(test_data_set, support_vectors.T)
    sigma = 1.0
    #K = np.dot(test_data_set, support_vectors.T)
    #kernel = build_kernel(train_data_set, kernel_type='rbf')
    c = (1. / sigma * np.sum(test_data_set ** 2, axis=1) * np.ones((1, np.shape(test_data_set)[0]))).T
    c = np.dot(c, np.ones((1, np.shape(kernel)[1])))
    #aa = np.dot((np.diag(K)*np.ones((1,len(test_data_set)))).T[support_vectors_id], np.ones((1, np.shape(K)[0]))).T
    sv = (np.diag(np.dot(train_data_set, train_data_set.T))*np.ones((1,len(train_data_set)))).T[support_vectors_id]
    aa = np.dot(sv,np.ones((1,np.shape(kernel)[0]))).T
    kernel = kernel - 0.5 * c - 0.5 * aa
    kernel = np.exp(kernel / (2. * sigma ** 2))

    y = np.zeros((np.shape(test_data_set)[0], 1))
    for j in range(np.shape(test_data_set)[0]):
        for i in range(vector_number):
            y[j] += lambdas[i] * targets[i] * kernel[j, i]
        y[j] += b
    return np.sign(y)

In [14]:
import numpy as np
from sklearn.model_selection import train_test_split

unique_labels = np.unique(iris.target)
decision_scores = []

for label in unique_labels:
    tr_data, te_data, tr_labels, te_labels = choose_set_for_label(iris.data, iris.target, label)

    objects_count = len(tr_labels)

    params = train(tr_data, tr_labels.reshape(-1, 1), kernel_type='rbf')
    score = classify_rbf(te_data, tr_data, *params)
    decision_scores.append(score)

predicted = np.argmax(np.column_stack(decision_scores), axis=1)
print(accuracy_score(te_labels, predicted))

     pcost       dcost       gap    pres   dres
 0:  1.3324e+02 -1.9748e+03  4e+03  2e-01  2e-15
 1:  8.6383e+01 -2.0657e+02  3e+02  8e-03  3e-15
 2:  1.1785e+01 -2.6208e+01  4e+01  1e-15  3e-15
 3: -6.8044e-01 -5.9075e+00  5e+00  4e-16  2e-15
 4: -1.8036e+00 -3.4676e+00  2e+00  2e-16  6e-16
 5: -2.2743e+00 -2.7856e+00  5e-01  5e-16  3e-16
 6: -2.3977e+00 -2.6013e+00  2e-01  5e-16  2e-16
 7: -2.4790e+00 -2.5651e+00  9e-02  3e-16  2e-16
 8: -2.5016e+00 -2.5138e+00  1e-02  2e-16  2e-16
 9: -2.5059e+00 -2.5061e+00  2e-04  3e-16  2e-16
10: -2.5060e+00 -2.5060e+00  5e-06  6e-16  2e-16
11: -2.5060e+00 -2.5060e+00  3e-07  6e-16  2e-16
Optimal solution found.


ValueError: shapes (30,4) and (1,11) not aligned: 4 (dim 1) != 1 (dim 0)